# T2 · One number to make small

**Tiny Transformer, Unit 2, module T2.** Every training step of the recorded run turned 2,048 right answers into one number, the
loss. Why minus the logs of their probabilities, and what does its size say? Here you remake the run's first batch at step 0 from
its seed and score it both ways, then score every held-back answer with the kept numbers: which answers make the loss big, what
counting right answers misses, and how many equal choices a loss is.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/T2-training-objective.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook T2-training-objective.ipynb`.

**Computed here:** the first batch at step 0, every held-back answer's probability and top pick, and what they add up to.
**Checked against the run:** its recorded first batch and first loss, its final held-back score, and the course's numbers, which
the notebook asserts.

Laid out like the page: two sections, four chapters, and under each chapter its page's three steps (Picture it, Watch it in our
run, Read and run the code), then the code to run and a "your turn".

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import base64, hashlib, json, math, os, urllib.request
import numpy as np
import torch

torch.set_num_threads(1)     # a model this small runs fastest on one CPU thread
print(torch.__version__)

2.10.0+cpu


**The text, the model and this module's code.** Tiny Shakespeare, pinned by its sha256; then the model's file, `model.py`, copied
in whole; then Unit 2's teaching functions, `code/training_steps.py`, copied in whole: T2's are the last six.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"{len(text):,} characters")

1,115,394 characters


In [4]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
#@title Unit 2's teaching functions: code/training_steps.py, copied whole (run me) { display-mode: "form" }
"""Unit 2 · Inside the learning loop: the recorded run's training, station by station, done explicitly.

The teaching functions of Unit 2 (each T notebook carries this file whole, and the chapter pages light its lines; the first
line is a Colab form title, so the notebooks fold this cell and no chapter's question is answered by its code early). They sit
beside M6's code/training_loop.py, with its names (get_batch, loss_of, train_step, every_window, average_loss), and repeat the
recorded run's own code, training/prompter/train.py: test_training_steps.py checks them against it and against the run's
numbers (training/prompter/unit2.py's export, prompter-unit2.json).
"""
import math
import os

import torch


# ---------------------------------------------------------------------------------------------------- T1 · picking the batches

def check_starts(n_train, n_val, batches=100, B=64, T=32, seed=2024):
    """The checks' fixed windows, as train.py draws them: 100 batches of starts in the training text, then 100 in the held-back
    text."""
    g = torch.Generator().manual_seed(seed)                       # the checks' generator
    train = torch.stack([torch.randint(n_train - T - 1, (B,), generator=g) for _ in range(batches)])   # [batches×B]
    val = torch.stack([torch.randint(n_val - T - 1, (B,), generator=g) for _ in range(batches)])       # [batches×B]
    return train, val


def run_starts(n, steps=100_000, B=64, T=32, seed=1337, make_model=None):
    """Every step's B window starts, as the recorded run drew them (train.py): the main random stream, started by the seed,
    first fills the model's random starting numbers (make_model builds the model), then draws B starts a step."""
    torch.manual_seed(seed)                                       # the main stream: starting numbers, then every batch
    if make_model is not None:
        make_model()                                              # first: the model's random starting numbers
    return torch.stack([torch.randint(n - T - 1, (B,)) for _ in range(steps)])   # [steps×B]  then B starts every step


def times_drawn(starts, n, T=32):
    """How many times each possible start was drawn. randint(n - T - 1) never draws its limit: starts run 0 … n - T - 2."""
    return torch.bincount(starts.flatten(), minlength=n - T - 1)  # [n−T−1]   one count per possible start


def times_asked(starts, n, T=32):
    """How many times each character of the text was a right answer: a window from start i asks about i + 1 … i + T."""
    drawn = times_drawn(starts, n, T)                             # [n−T−1]
    change = torch.zeros(n + 1, dtype=torch.long)
    change[1:n - T] += drawn                                      # a window from i starts asking at i + 1 …
    change[T + 1:n] -= drawn                                      # … and stops after i + T
    return change.cumsum(0)[:n]                                   # [n]       one count per character


def times_through(n, steps=100_000, B=64, T=32):
    """How many times over the run reads its text: every step asks B × T questions, one per right answer."""
    return steps * B * T / n                                      # answers ÷ characters


# ---------------------------------------------------------------------------------------------------- T2 · the loss

def all_at_once(model, x, y):
    """A batch's right answers all at once: their probabilities multiplied, in the model's 32-bit numbers, and their costs,
    minus each one's log, added and averaged (the loss)."""
    probs = torch.softmax(model(x)[0], dim=-1)                    # [B×T×65]  probabilities, as M6's loss_of makes them
    p = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)             # [B×T]     each right answer's probability
    product = p.prod()                                            # []        all of them together: too small for 32 bits, 0.0
    total = -p.log().sum()                                        # []        their costs, added
    return product, total, total / p.numel()                      # the product, the sum, and the average: the loss


def loss_from_scores(model, x, y):
    """The loss as model.py's F.cross_entropy works it out, straight from the scores: for each answer, the log of the total of
    e to every score, minus the right answer's own score. No probability is ever made."""
    scores = model(x)[0]                                          # [B×T×65]  a score for every character
    right = scores.gather(-1, y.unsqueeze(-1)).squeeze(-1)        # [B×T]     each right answer's score
    return (scores.logsumexp(-1) - right).mean()                  # []        minus the log of each probability, averaged


def answer_costs(model, ids, T=32, per=256):
    """Every right answer's probability over a whole text, and whether it was the top pick, read as the final score reads it
    (train.py full_loss): windows side by side, every character scored once but the first and the last few, which fill no
    window."""
    n = len(range(0, len(ids) - T - 1, T))                        # how many whole windows fit
    x, y = ids[:n * T].view(n, T), ids[1:n * T + 1].view(n, T)    # [n×T] each: the inputs, and the targets moved one along
    p, top = [], []
    with torch.no_grad():
        for xb, yb in zip(x.split(per), y.split(per)):
            probs = torch.softmax(model(xb)[0], dim=-1)           # [256×T×65]
            p.append(probs.gather(-1, yb.unsqueeze(-1)).squeeze(-1))   # [256×T]  the right answer's probability
            top.append(probs.argmax(-1) == yb)                    # [256×T]  was it the top pick?
    return torch.cat(p).flatten(), torch.cat(top).flatten()       # [n·T] each


def share_of_loss(p, below=0.01):
    """The answers given a probability under `below`: their share of all the answers, and their share of the loss."""
    cost = -p.log()                                               # each answer's cost
    rare = p < below
    return rare.float().mean().item(), (cost[rare].sum() / cost.sum()).item()


def sure_or_right(p, top):
    """Accuracy, the share of answers whose top pick is right; and the share right but unsure: top pick, yet under one half."""
    return top.float().mean().item(), (top & (p < 0.5)).float().mean().item()


def choices(loss):
    """A loss read backwards: an even guess among n characters costs ln n on every answer, so n = e to the loss."""
    return math.exp(loss)


# ---------------------------------------------------------------------------------------------------- T3 · the way back

def bias_gradient(probs, y):
    """Every readout bias's gradient, by its rule. A bias adds to one character's score at every place, so its gradient is that
    character's probability averaged over the places, minus its share of the right answers."""
    V = probs.size(-1)
    predicted = probs.reshape(-1, V).mean(0)                      # [65]  each character's average probability
    right = torch.bincount(y.flatten(), minlength=V).float() / y.numel()   # [65]  its share of the right answers
    return predicted - right                                      # [65]  average probability, minus share of right answers


def nudge(model, x, y, param, i, h=1e-3):
    """One number's gradient without backpropagation: nudge it up by h and run the batch, then down by h and run it again, and
    take the slope through the two losses. The number is put back as it was."""
    flat = param.data.view(-1)
    old = flat[i].item()
    with torch.no_grad():
        flat[i] = old + h
        up = model(x, y)[1].item()                                # the loss, nudged up
        flat[i] = old - h
        down = model(x, y)[1].item()                              # the loss, nudged down
        flat[i] = old                                             # put back
    return (up - down) / (2 * h)


def grad_twice(model, x, y, param):
    """backward fills every learned number's .grad in one pass, and moves none of them. Run it again without clearing, and the
    gradients add up: why train.py clears them first, with opt.zero_grad()."""
    model.zero_grad()
    model(x, y)[1].backward()                                     # every .grad filled; no number moved
    once = param.grad.clone()
    model(x, y)[1].backward()                                     # again, without clearing
    return once, param.grad.clone()                               # the second: twice the first


def backward_sizes(model, x, y):
    """The gradient on its way back through model.py's pass, part by part: its size (all its numbers measured together) at the
    scores, at the block's top, into each add's branch, below each add, and at the rows the tables made."""
    blk = model.blocks[0]
    rows = model.tok_emb(x) + model.pos_emb(torch.arange(x.size(1)))   # [B×T×28]  token row + position row
    att = blk.attn(blk.ln1(rows))                                 # the attention's branch
    mid = rows + att                                              # its add
    ff = blk.ff(blk.ln2(mid))                                     # the feed-forward's branch
    top = mid + ff                                                # its add: the block's top
    scores = model.ln_f(top) @ model.tok_emb.weight.T + model.head_bias   # [B×T×65]  the readout
    loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
    parts = {"scores": scores, "top": top, "ff_branch": ff, "meet": mid, "attn_branch": att, "rows": rows}
    grads = torch.autograd.grad(loss, list(parts.values()))       # every part's gradient, in one pass back
    return {name: g.norm().item() for name, g in zip(parts, grads)}, dict(zip(parts, grads))


def table_gradient_by_use(model, x, y):
    """The token table's gradient, one job at a time: read in at the start (each input's entry), and reused at the readout (scoring
    every character). Holding one use still (detach) leaves the other's push; the two add up to the whole gradient."""
    W = model.tok_emb.weight                                      # [65×28]

    def push(use):
        rows = (W if use == "in" else W.detach())[x] + model.pos_emb(torch.arange(x.size(1)))
        for b in model.blocks:
            rows = b(rows)
        scores = model.ln_f(rows) @ (W if use == "out" else W.detach()).T + model.head_bias
        loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
        return torch.autograd.grad(loss, W)[0]                    # [65×28]

    return push("in"), push("out")


# ---------------------------------------------------------------------------------------------------- T4 · AdamW

def adamw_step(p, g, m, v, t, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    """One AdamW step for any tensor of numbers, as torch.optim.AdamW takes it on PyTorch's defaults (train.py passes only the
    learning rate). m and v are the running averages of the pushes and of their squares, both 0 before step 1; t counts the
    steps from 1. Returns the moved numbers and the two averages."""
    b1, b2 = betas
    p = p * (1 - lr * weight_decay)                               # the W: weight decay, a pull toward 0 (T4.4)
    m = b1 * m + (1 - b1) * g                                     # the pushes, averaged with their signs (T4.3)
    v = b2 * v + (1 - b2) * g * g                                 # their sizes, squared and averaged
    m_hat = m / (1 - b1 ** t)                                     # both start at 0: the corrections (T4.2)
    v_hat = v / (1 - b2 ** t)
    step = m_hat / (v_hat.sqrt() + eps)                           # each push over its own size: about ±1 at step 1 (T4.1)
    return p - lr * step, m, v


def steps_in_rates(pushes, betas=(0.9, 0.999), eps=1e-8, corrected=True):
    """One number's move at each step, in learning rates (weight decay aside), for a list of its pushes."""
    b1, b2 = betas
    m = v = 0.0
    moves = []
    for t, g in enumerate(pushes, 1):
        m, v = b1 * m + (1 - b1) * g, b2 * v + (1 - b2) * g * g
        if corrected:
            moves.append(-(m / (1 - b1 ** t)) / (math.sqrt(v / (1 - b2 ** t)) + eps))
        else:
            moves.append(-m / (math.sqrt(v) + eps))               # without the corrections
    return moves


def decay_left(rates, weight_decay=0.01):
    """What weight decay alone leaves of a number: each step keeps 1 − weight_decay × that step's learning rate."""
    left = 1.0
    for lr in rates:
        left *= 1 - weight_decay * lr
    return left


def learning_rate(step, lr=3e-3, min_lr=3e-4, warmup=200, max_steps=100_000):
    """train.py's schedule (lr_at): a straight climb to lr over the warm-up, then down a cosine to min_lr at the last step."""
    if step < warmup:
        return lr * step / warmup
    done = min(1.0, (step - warmup) / (max_steps - warmup))
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * done))


def first_steps(model, train_ids, steps, rate=learning_rate, B=64, T=32, watch=None):
    """The run's first steps, as train.py takes them (the seed set just before the model was made): each step's learning rate,
    64 random windows, their loss, the way back, then AdamW's step. Calls watch(model) after every step; returns its answers."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    seen = []
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = rate(step + 1)                          # this step's learning rate
        starts = torch.randint(len(train_ids) - T - 1, (B,))      # this step's 64 places
        x = torch.stack([train_ids[i:i + T] for i in starts])
        y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
        loss = model(x, y)[1]
        opt.zero_grad(set_to_none=True)
        loss.backward()                                           # the way back
        opt.step()                                                # AdamW moves every number
        if watch:
            seen.append(watch(model))
    return seen


@torch.no_grad()
def check_loss(model, ids, starts, T=32):
    """The run's check: the average loss over fixed windows (check_starts), as train.py scores it every 500 steps."""
    model.eval()
    total = sum(model(torch.stack([ids[i:i + T] for i in s]), torch.stack([ids[i + 1:i + T + 1] for i in s]))[1].item() for s in starts)
    model.train()
    return total / len(starts)


# ---------------------------------------------------------------------------------------------------- T5 · the loop

def one_turn(model, opt, train_ids, step, B=64, T=32, clear=True):
    """One turn of train.py's loop, in its order: this step's learning rate, 64 random windows, their loss, the old gradients
    cleared, the way back, then AdamW's step. clear=False leaves the old gradients in (T5.1). Returns the turn's loss."""
    for group in opt.param_groups:
        group["lr"] = learning_rate(step + 1)                     # 1 · this step's learning rate
    starts = torch.randint(len(train_ids) - T - 1, (B,))          # 2 · 64 random places
    x = torch.stack([train_ids[i:i + T] for i in starts])
    y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
    loss = model(x, y)[1]                                         # 3 · their loss
    if clear:
        opt.zero_grad(set_to_none=True)                           # 4 · the last turn's gradients cleared
    loss.backward()                                               # 5 · the way back adds each gradient to what .grad holds
    opt.step()                                                    # 6 · AdamW moves every number
    return loss.item()


def run_turns(model, train_ids, steps, check=None, every=50, clear=True, remake=False):
    """Turns 1 to `steps` from where the model is, scoring check(model) every `every` steps. remake=True makes a new AdamW every
    turn, so its averages and its step count are lost (T5.2). Returns [[step, check], ...]."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    curve = []
    for step in range(steps + 1):
        if check and step % every == 0:
            curve.append([step, check(model)])
        if step == steps:
            return curve
        if remake:
            opt = torch.optim.AdamW(model.parameters(), lr=3e-3)  # a new AdamW: its averages start at 0 again
        one_turn(model, opt, train_ids, step, clear=clear)


def what_lasts(model, opt):
    """What carries on from one turn to the next: the learned numbers, AdamW's two averages for each, and its step count."""
    numbers = sum(p.numel() for p in model.parameters())
    averages = sum(s["exp_avg"].numel() + s["exp_avg_sq"].numel() for s in opt.state.values())
    steps = sorted({int(s["step"]) for s in opt.state.values()})
    return numbers, averages, steps


def saved(path):
    """What a saved file holds: each table's name and shape, how many numbers in all, and the file's size in bytes."""
    tables = torch.load(path, weights_only=True)
    shapes = {name: tuple(t.shape) for name, t in tables.items()}
    return shapes, sum(t.numel() for t in tables.values()), os.path.getsize(path)


# ---------------------------------------------------------------------------------------------------- T6 · the check

def other_checks(model, ids, seeds, batches=100, B=64, T=32):
    """The check on other windows: for each seed, its own 100 batches of 64 starts, drawn as train.py draws the check's, scored
    as check_loss scores them. The model's numbers never change: only the windows do."""
    scores = []
    for seed in seeds:
        g = torch.Generator().manual_seed(seed)                   # another seed: other windows
        starts = [torch.randint(len(ids) - T - 1, (B,), generator=g) for _ in range(batches)]
        scores.append(check_loss(model, ids, starts, T))
    return scores                                                 # one score per seed


def how_often_asked(starts, n, T=32):
    """How many characters of a text the check's windows ask about 0 times, once, twice, …: times_asked over those windows,
    counted by times. The first and the last characters are never asked: no window starts before the first or reaches the
    last."""
    asked = times_asked(starts, n, T)                             # [n]   one count per character
    return torch.bincount(asked).tolist()                         # [how many 0 times, once, twice, …]


def loss_by_position(model, ids, T=32):
    """The final score, position by position: answer_costs reads the whole text in windows of 32, side by side; each
    position's costs are averaged down the windows. Position 0 has one character to go on, position 31 all 32."""
    p, _ = answer_costs(model, ids, T)                            # [n·T]  every right answer's probability, window after window
    return (-p.log()).view(-1, T).mean(0)                         # [T]    one average cost per position


def keep_best(checks, margin=1e-4, patience=20):
    """train.py's rule, replayed over a log of checks ([step, held-back score] pairs): a check that beats the best so far by
    more than `margin` keeps a copy, and the wait starts again; any other check adds 1 to the wait; a wait of `patience`
    would stop the run. Returns the kept steps, the step it would stop at (None: never), and the kept copy's score."""
    best, wait, kept = float("inf"), 0, []
    for step, score in checks:
        if score < best - margin:
            best, wait = score, 0                                 # a new best: keep a copy
            kept.append(step)
        elif step:
            wait += 1                                             # no new best: one more check waited
        if patience and wait >= patience:
            return kept, step, best                               # train.py would stop the run here
    return kept, None, best

In [6]:
tok = Tokenizer(text)
cfg = Config(vocab_size=len(tok.chars))
ids = torch.tensor(tok.encode(text))                        # the whole text as ids
cut = int(0.9 * len(ids))
train_ids, val_ids = ids[:cut], ids[cut:]                   # the first 90% trains; the last 10% is held back

# step 0, in train.py's order: the seed, then the model (its starting numbers), then the first batch's 64 starts
torch.manual_seed(1337)
model0 = Prompter(cfg)
starts = torch.randint(len(train_ids) - 32 - 1, (64,))
x0 = torch.stack([train_ids[i:i + 32] for i in starts])        # [64×32]  the inputs
y0 = torch.stack([train_ids[i + 1:i + 33] for i in starts])    # [64×32]  their right answers
print('the first batch:', tuple(x0.shape), ' its first starts:', starts[:4].tolist())
assert starts.tolist() == [160204, 793724, 810574, 964722, 40734, 423895, 818722, 483307, 510490, 928356, 614059, 30062, 737496, 407796, 59365, 998494, 864171, 929951, 942896, 637372, 408276, 289366, 811280, 165163, 748492, 428999, 260643, 560439, 371836, 390719, 776246, 678592, 338694, 316420, 711307, 461584, 554475, 251253, 546994, 358001, 64331, 976218, 589122, 206049, 243936, 44076, 936280, 819204, 162295, 566156, 935951, 895342, 457178, 537830, 274346, 823229, 14080, 899619, 528867, 495983, 90293, 96640, 21128, 975278]   # the recorded run's first batch

# the kept numbers: the run's best check, step 98,000 (the course's prompter.json, the same numbers as best.pt)
paths = [os.path.join(here, '..', 'model', 'prompter.json'),                                                     # in the labs
         os.path.join(here, '..', '..', '..', '..', 'src', 'specimen', 'tiny-transformer', 'data', 'prompter.json')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the model's trained numbers are not here: run the Setup cell at the top"
exp = json.load(open(found[0]))
state = {t['name']: torch.tensor(np.frombuffer(base64.b64decode(t['data']), dtype='<f4').reshape(t['shape']).copy())
         for t in exp['weights']['tensors']}
model = Prompter(cfg)
model.load_state_dict(state)
model.eval()
print(f"the kept numbers: run {exp['run']}, step {exp['checkpoint_step']:,}, {count_params(model):,} learned numbers")
assert count_params(model) == 9417 and exp['checkpoint_step'] == 98000

the first batch: (64, 32)  its first starts: [160204, 793724, 810574, 964722]
the kept numbers: run prompter-r1, step 98,000, 9,417 learned numbers


# Section 1 · Why minus the log

Why the loss is built from minus logs, and which answers make it big. Each chapter below follows its page: Picture it, Watch it in
our run, then Read and run the code.

## T2.1 · All the answers at once

*Why does the loss add minus logs, instead of multiplying probabilities?*

**Predict first** (the page asks it too, before its clip): at step 0, each of the first batch's 2,048 right answers got a probability of about 1 in 65. Multiply all 2,048 together: can
the computer store the result? No, it is far too small, and would be stored as 0; yes, a computer can store any small number; or
only just: it is close to the smallest a computer can hold?

### T2.1.1 · Picture it

A batch does well when all its right answers are likely together. Together, their probability is each one's, multiplied. Picture 4
made-up right answers, with probabilities ½, ¼, ½ and ⅛. Multiplied: 1 in 128, smaller with every answer. Minus the log turns each
into a cost (0.69, 1.39, 0.69 and 2.08). The costs add to 4.85: minus the log of the product. (The page's answers are made up to
show the idea.)

### T2.1.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is the run's first batch, at step 0: 2,048 right answers, each given about 1 in 65. Multiplied, their probability all at
once has 3,716 zeros after the decimal point before its first digit. The model's 32-bit numbers hold nothing past 44
zeros. Stored, that product would be exactly 0. Their costs add instead: 2,048 minus logs make 8,557. Divided by
2,048, that is 4.18, the first loss M6 showed.

### T2.1.3 · Read and run the code

`all_at_once` in `code/training_steps.py` (copied in above) scores the first batch both ways. It picks each right answer's
probability, as M6's `loss_of` did. Then `torch.prod` multiplies all 2,048: 0.0. Their minus logs, added, make
8,557; averaged, 4.18, the same as `model.py`'s own loss. `loss_from_scores` works as `model.py`'s
`F.cross_entropy` does, straight from the scores: the log of the total of e to every score, minus the right one's. No probability
is ever made. PyTorch calls this loss cross-entropy.

In [7]:
with torch.no_grad():
    product, total, mean = all_at_once(model0, x0, y0)     # the first batch at step 0, both ways
    own, scores = model0(x0, y0)[1].item(), loss_from_scores(model0, x0, y0).item()
print('multiplied:', product.item(), ' added:', round(total.item(), 2), ' averaged:', round(mean.item(), 4))
print("model.py's own loss:", round(own, 4), ' straight from the scores:', round(scores, 4))
print('zeros after the point in the product:', math.floor(total.item() / math.log(10)),
      ' in the smallest 32-bit number:', math.floor(-math.log10(np.finfo(np.float32).smallest_subnormal)))
assert product.item() == 0.0 and abs(total.item() - 8556.78) < 0.05 and abs(mean.item() - 4.17812) < 1e-4
assert abs(own - mean.item()) < 1e-5 and abs(scores - own) < 1e-5
assert math.floor(total.item() / math.log(10)) == 3716 and math.floor(-math.log10(np.finfo(np.float32).smallest_subnormal)) == 44

multiplied: 0.0  added: 8556.78  averaged: 4.1781
model.py's own loss: 4.1781  straight from the scores: 4.1781
zeros after the point in the product: 3716  in the smallest 32-bit number: 44


### ✋ Your turn

Multiply the first batch's probabilities one at a time, in 32-bit numbers: after how many answers is the product 0? Take each right
answer's probability, `p0`, as `all_at_once` does, and count the products still above 0 with `torch.cumprod`. Put the count in
`answer`.

In [8]:
with torch.no_grad():
    p0 = torch.softmax(model0(x0)[0], dim=-1).gather(-1, y0.unsqueeze(-1)).flatten()   # the 2,048 probabilities
answer = None   # TODO: (torch.cumprod(p0, 0) > 0).sum(), then int(...)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int((torch.cumprod(p0, 0) > 0).sum()), 'the running products: torch.cumprod(p0, 0); count those above 0: int((... > 0).sum())'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [9]:
#@title Show a solution { display-mode: "form" }
with torch.no_grad():
    p0 = torch.softmax(model0(x0)[0], dim=-1).gather(-1, y0.unsqueeze(-1)).flatten()
answer = int((torch.cumprod(p0, 0) > 0).sum())
print(f"{answer} answers multiplied, then 0: at about 1/65 each, 45 zeros take about 45 / 1.81 = 25 answers")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int((torch.cumprod(p0, 0) > 0).sum()), 'the running products: torch.cumprod(p0, 0); count those above 0: int((... > 0).sum())'
    print("✓ right:", answer)

24 answers multiplied, then 0: at about 1/65 each, 45 zeros take about 45 / 1.81 = 25 answers
✓ right: 24


## T2.2 · Where the loss comes from

*Which answers make the loss big?*

**Predict first** (the page asks it too, before its clip): about 5.5 in every 100 held-back answers got a probability under 0.01. What share of the whole loss do they make:
about 18 in 100; about 11 in 100, twice their share of the answers; or more than half of it?

### T2.2.1 · Picture it

The loss is an average of costs. Does every right answer weigh the same in it? Picture 10 made-up right answers, sorted by
probability, each with its cost: minus its log. The one given 0.01 costs 4.61: more than the 7 surest together. A cost has no
ceiling: the less likely a right answer was, the more it costs. (The page's answers are made up to show the idea.)

### T2.2.2 · Watch it in our run

*(The page's recorded clip, in words.)*

The best check's kept numbers scored all 111,520 held-back answers. Here they are, by what each cost. The middle answer got a
probability of 0.19. Many cost little, near 0; the costs trail off far to the right. Past 4.61, a probability under
0.01: 5.5 in every 100 answers. Together they make 18 in every 100 of the loss. A few bad
misses weigh far more than their number: the loss counts how badly each answer was missed.

### T2.2.3 · Read and run the code

`answer_costs` reads the held-back text as the run's final score did, in windows side by side. It keeps every right answer's
probability: 111,520 of them. Their average cost is 1.93, the final score. `share_of_loss` picks the
answers under 0.01, and adds up their costs, minus each log. It takes a few seconds.

In [10]:
p, top = answer_costs(model, val_ids)                      # every held-back answer: its probability, and was it the top pick
cost = -p.log()
low, low_loss = share_of_loss(p, 0.01)
print(f"{p.numel():,} answers · average cost {cost.mean():.4f} · the middle probability {p.median():.2f}")
print(f"under 0.01: {100 * low:.1f} in 100 answers · {100 * low_loss:.0f} in 100 of the loss")
assert p.numel() == 111520 and abs(cost.mean().item() - 1.9317) < 1e-4   # the run's final held-back score
assert abs(p.median().item() - 0.186617) < 1e-4
assert abs(low - 0.0545732) < 1e-5 and abs(low_loss - 0.178585) < 1e-5

111,520 answers · average cost 1.9317 · the middle probability 0.19
under 0.01: 5.5 in 100 answers · 18 in 100 of the loss


### ✋ Your turn

Now a wider net: the answers given under 0.1. What share of the loss do they make? Use `share_of_loss` with `below=0.1`, and put
the loss share (from 0 to 1) in `answer`.

In [11]:
answer = None   # TODO: share_of_loss(p, 0.1)[1]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - share_of_loss(p, 0.1)[1]) < 1e-6, 'the second number share_of_loss gives: share_of_loss(p, 0.1)[1]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [12]:
#@title Show a solution { display-mode: "form" }
answer = share_of_loss(p, 0.1)[1]
print(f"under 0.1: {100 * share_of_loss(p, 0.1)[0]:.1f} in 100 answers, {100 * answer:.1f} in 100 of the loss")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - share_of_loss(p, 0.1)[1]) < 1e-6, 'the second number share_of_loss gives: share_of_loss(p, 0.1)[1]'
    print("✓ right:", answer)

under 0.1: 36.6 in 100 answers, 68.6 in 100 of the loss
✓ right: 0.6859372854232788


# Section 2 · Reading the number

What the loss says, and what counting right answers misses.

## T2.3 · Right, but not sure

*Does counting right answers tell the same story as the loss?*

**Predict first** (the page asks it too, before its clip): of the held-back answers whose top pick is the right character, how many got a probability under one half: over a third of them;
only a few in 100, since a right top pick is nearly always over one half; or almost all of them?

### T2.3.1 · Picture it

Another score: count the answers whose top pick, the most probable character, is right. Does it agree with the loss? Picture 3
made-up answers, the right character marked. The first gives the right character 0.9; the second, 0.4, still the top pick. The
third picks another (0.3 against 0.5). Counting top picks, the first two are equally right. The loss charges 0.11 and 0.92: being
sure counts too. (The page's answers are made up to show the idea.)

### T2.3.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are the held-back answers again, as 100 squares: what the kept numbers' top pick did. For 44 in every 100, the
top pick is the right character. That share is called accuracy. But 16 of those 44 got under one
half: right, yet unsure. Only 28 in 100 were right and over one half. Accuracy counts every right top pick as fully
right. The loss also charges for doubt: it rewards being right and sure.

### T2.3.3 · Read and run the code

`answer_costs` also marks each top pick, with `argmax`, the most probable character. The mark is true where that character is the
right answer. `sure_or_right` averages those marks: the accuracy. Then it keeps only the right ones under one half.

In [13]:
acc, unsure = sure_or_right(p, top)
print(f"accuracy: {100 * acc:.0f} in 100 · right yet under one half: {100 * unsure:.0f} · right and over: {100 * (acc - unsure):.0f}")
assert abs(acc - 0.438755) < 1e-5 and abs(unsure - 0.161442) < 1e-5
assert bool(((p > 0.5) <= top).all())                       # over one half is always the top pick

accuracy: 44 in 100 · right yet under one half: 16 · right and over: 28


### ✋ Your turn

Of the right top picks, what fraction got under one half? Use `acc` and `unsure`, and put the fraction (from 0 to 1) in `answer`.
Is it over a third?

In [14]:
answer = None   # TODO: unsure / acc

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - 0.367955) < 1e-4, 'the unsure ones over all the right ones: unsure / acc'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [15]:
#@title Show a solution { display-mode: "form" }
answer = unsure / acc
print(f"{answer:.3f} of the right top picks were under one half: over a third, {answer > 1 / 3}")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - 0.367955) < 1e-4, 'the unsure ones over all the right ones: unsure / acc'
    print("✓ right:", answer)

0.368 of the right top picks were under one half: over a third, True
✓ right: 0.3679542062633625


## T2.4 · How many choices?

*What does the loss's size mean?*

**Predict first** (the page asks it too, before its clip): our model's held-back loss is 1.93. An even guess among how many characters would score the same: about 7; about
85, 10 to the power 1.93; or about 33, half of 65, as 1.93 is about half of 4.17?

### T2.4.1 · Picture it

An even guess among n characters costs the natural log of n. Can a loss be read backwards? Picture the 4 made-up answers again
(½, ¼, ½ and ⅛). Their costs average 1.21. An even guess among 3.4 costs the same: e to the power 1.21 is 3.4. So a loss reads as
equal choices: these 4 are as unsure as a pick among 3.4. (The page's answers are made up to show the idea.)

### T2.4.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are M1's three scores on the held-back text, read backwards as even guesses. The even guess scored 4.17: all 65
characters equally likely. Counting pairs scored 2.48: as unsure as an even guess among 12. Our model
scored 1.93: among 6.9. So after all its training, our model is, on average, as unsure as a pick among
about 7 characters. A loss of 0 would be 1 choice: sure, and right.

### T2.4.3 · Read and run the code

`choices` is one line, e to the power of the loss. The cell averages `answer_costs` over the held-back text, 1.93,
the final score; e to the 1.93 is 6.9. Counting pairs gives 12; the even guess,
exactly 65. (Textbooks call e to the loss perplexity.)

In [16]:
held = cost.mean().item()                                   # the held-back loss: the run's final score
PAIRS = 2.4819                                       # counting pairs on the held-back text (M1), recorded
for name, l in (('the even guess', math.log(65)), ('counting pairs', PAIRS), ('our model', held), ('sure and right', 0.0)):
    print(f"{name:>15}: loss {l:.2f} → {choices(l):.1f} equal choices")
assert abs(choices(held) - 6.90121) < 1e-3 and abs(choices(PAIRS) - 11.964) < 1e-3
assert abs(choices(math.log(65)) - 65) < 1e-9 and choices(0.0) == 1.0

 the even guess: loss 4.17 → 65.0 equal choices
 counting pairs: loss 2.48 → 12.0 equal choices
      our model: loss 1.93 → 6.9 equal choices
 sure and right: loss 0.00 → 1.0 equal choices


### ✋ Your turn

The first batch at step 0 scored 4.18. As unsure as an even guess among how many characters? Use `choices` on
`mean` (from T2.1), and put the number in `answer`. Is it near 65?

In [17]:
answer = None   # TODO: choices(mean.item())

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - 65.243081) < 0.01, 'e to the first loss: choices(mean.item())'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [18]:
#@title Show a solution { display-mode: "form" }
answer = choices(mean.item())
print(f"step 0: as unsure as an even guess among {answer:.1f}: a little worse than the even guess, 65")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - 65.243081) < 0.01, 'e to the first loss: choices(mean.item())'
    print("✓ right:", answer)

step 0: as unsure as an even guess among 65.2: a little worse than the even guess, 65
✓ right: 65.24277863968211


The loss never multiplies: 2,048 probabilities multiplied are too small to store, so it adds their minus logs, one number a
computer can hold. A few bad misses make much of it, it charges for doubt that counting right answers never sees, and its size
reads as a choice: 1.93 held back is, on average, about 7 equal choices.

Next: **T3 · the way back**, from that one number to every learned number.